In [1]:
!git clone https://github.com/ChrisSkjer/energy-stable-pinns-ch.git
%cd energy-stable-pinns-ch

Cloning into 'energy-stable-pinns-ch'...
remote: Enumerating objects: 326, done.
remote: Counting objects: 100% (326/326), done.
remote: Compressing objects: 100% (216/216), done.
remote: Total 326 (delta 189), reused 237 (delta 100), pack-reused 0 (from 0)
Receiving objects: 100% (326/326), 6.20 MiB | 15.01 MiB/s, done.
Resolving deltas: 100% (189/189), done.
/content/energy-stable-pinns-ch


In [2]:
!pip install -r requirements-colab.txt

In [3]:
import torch
print(torch.__version__, "| cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

2.11.0+cu128 | cuda available: True
Tesla T4


Flags for `python -m src.pinn.train` (defaults shown; patience 0 = off):
```
model    --hidden-layers 5  --hidden-width 100
adam     --epochs 100  --lr 1e-3  --adam-tol 1e-4  --adam-patience 0
plateau  --plateau-patience 0  --plateau-factor 0.5  --min-lr 1e-6
lbfgs    --lbfgs-steps 70  --lbfgs-tol 1e-6  --lbfgs-patience 3
weights  --pde-u-weight 1  --pde-mu-weight 100  --ic-weight 100  --bc-weight 10
energy   --energy-penalty (switch)  --energy-weight 1
problem  --epsilon 0.05  --x-max 1  --y-max 1  --t-max 5e-3
points   --n-collocation 10000  --n-ic 100  --n-bc 100  --pde-at-t0 2000
run      --device cpu  --run-name <timestamp>  --checkpoint-every 500  --log-every 100  --smoke-test
```

In [4]:
run_name = "eps0.05_h100x5_t0_0.01_p20000_wmu300_icw1000_nopdeat-t0"  # set once here -- reused below for training and the downloaded zip
!python -m src.pinn.train --run-name {run_name} --device cuda --epochs 2000 --checkpoint-every 500 --lbfgs-steps 400 --epsilon 0.05 --t-max 0.01 --n-collocation 20000 --pde-at-t0 0 --ic-weight 1000 --pde-mu-weight 300

run directory: results/pinn_models/eps0.05_h100x5_t0_0.01_p20000_wmu300_icw1000_nopdeat-t0
[adam] epoch     0/2000 lr 1.00e-03 pde_u 7.4396e+01 pde_mu 3.0076e+01 ic 7.3231e+02 bc 3.8018e+00 total 8.4058e+02
[adam] epoch   100/2000 lr 1.00e-03 pde_u 9.2817e-01 pde_mu 3.4313e-02 ic 3.4556e+02 bc 5.8155e-02 total 3.4658e+02
[adam] epoch   200/2000 lr 1.00e-03 pde_u 2.2928e+00 pde_mu 1.3407e-01 ic 3.2691e+02 bc 2.5113e-01 total 3.2959e+02
[adam] epoch   300/2000 lr 1.00e-03 pde_u 6.0065e+00 pde_mu 1.2651e+00 ic 1.8857e+02 bc 3.0537e+00 total 1.9890e+02
[adam] epoch   400/2000 lr 1.00e-03 pde_u 3.0841e+00 pde_mu 9.7000e+00 ic 7.0341e+01 bc 3.4205e+00 total 8.6546e+01
[adam] epoch   500/2000 lr 1.00e-03 pde_u 1.2658e+00 pde_mu 8.1593e+00 ic 6.2011e+01 bc 2.6240e+00 total 7.4060e+01
[adam] epoch   600/2000 lr 1.00e-03 pde_u 8.8115e-01 pde_mu 6.9572e+00 ic 5.5796e+01 bc 2.5858e+00 total 6.6220e+01
[adam] epoch   700/2000 lr 1.00e-03 pde_u 9.9315e-01 pde_mu 5.8467e+00 ic 4.8279e+01 bc 2.4423e+0

In [5]:
# NOTE: google.colab.files.download() needs the actual colab.research.google.com
# frontend to receive the browser download -- it silently does nothing when this
# notebook is driven from VS Code's Jupyter extension against a Colab kernel.
# Instead, embed the model as base64 text in the cell output, which VS Code
# stores locally in this .ipynb file once you save it (Ctrl+S). Then run
# `python scripts/pull_colab_model.py` locally to decode it.
import base64
import pathlib

model_path = pathlib.Path(f"results/pinn_models/{run_name}/final.pt")
b64 = base64.b64encode(model_path.read_bytes()).decode()
print(f"===BEGIN_MODEL_B64:{run_name}/final.pt===")
print(b64)
print("===END_MODEL_B64===")

===BEGIN_MODEL_B64:eps0.05_h100x5_t0_0.01_p20000_wmu300_icw1000_nopdeat-t0/final.pt===
...
===END_MODEL_B64===
